### Cell 11.01 — imports, project paths, and output folders

In [ ]:
# Cell 11.01
# Manuscript figures notebook
# This notebook uses ONLY frozen analytical outputs.
# No linkage-map reconstruction, QTL rescanning, permutation testing,
# or candidate-gene reprioritization is performed here.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()


# Frozen Notebook 10 outputs
TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
)

SOURCE_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)


# Dedicated Notebook 11 figure outputs
MANUSCRIPT_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "manuscript"
)

MANUSCRIPT_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Dedicated source-data tables for manuscript figures
MANUSCRIPT_FIGURE_TABLE_DIR = (
    TABLE_DIR
    / "manuscript_figures"
)

MANUSCRIPT_FIGURE_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


NOTEBOOK10_FILE = (
    TABLE_DIR
    / "flyer_hartwig_notebook10_final_synthesis.xlsx"
)


print("NOTEBOOK 11 — MANUSCRIPT FIGURES")
print("=" * 100)

print("Project root:")
print(PROJECT_ROOT)

print()
print("Frozen Notebook 10 workbook:")
print(NOTEBOOK10_FILE)

print()
print("Manuscript figure directory:")
print(MANUSCRIPT_FIGURE_DIR)

print()
print("Figure source-table directory:")
print(MANUSCRIPT_FIGURE_TABLE_DIR)

### Cell 11.02 — verify frozen inputs and load core tables

In [ ]:
# Cell 11.02
# Verify and load the frozen Notebook 10 synthesis workbook.

if not NOTEBOOK10_FILE.exists():
    raise FileNotFoundError(
        f"Frozen Notebook 10 workbook not found:\n{NOTEBOOK10_FILE}"
    )


xls = pd.ExcelFile(
    NOTEBOOK10_FILE
)


print("FROZEN NOTEBOOK 10 SHEETS")
print("=" * 100)

for sheet in xls.sheet_names:
    print(" -", sheet)


# ----------------------------------------------------------
# Load core frozen tables
# ----------------------------------------------------------

table1_qtl = pd.read_excel(
    NOTEBOOK10_FILE,
    sheet_name="Table1_main_qtl"
)

all33_qtl = pd.read_excel(
    NOTEBOOK10_FILE,
    sheet_name="TableS1_all33_qtl"
)

four_qtl_full = pd.read_excel(
    NOTEBOOK10_FILE,
    sheet_name="four_qtl_full"
)

peak_effects = pd.read_excel(
    NOTEBOOK10_FILE,
    sheet_name="peak_allele_effects"
)

candidate_evidence = pd.read_excel(
    NOTEBOOK10_FILE,
    sheet_name="candidate_evidence"
)

map_fragments = pd.read_excel(
    NOTEBOOK10_FILE,
    sheet_name="map_fragments"
)

physical_assignments = pd.read_excel(
    NOTEBOOK10_FILE,
    sheet_name="physical_assignments"
)


print()
print("LOADED FROZEN TABLES")
print("=" * 100)

print("Table 1 QTL:", table1_qtl.shape)
print("All 33 QTL:", all33_qtl.shape)
print("Four QTL full:", four_qtl_full.shape)
print("Peak effects:", peak_effects.shape)
print("Candidate evidence:", candidate_evidence.shape)
print("Map fragments:", map_fragments.shape)
print("Physical assignments:", physical_assignments.shape)

### Cell 11.03 — validation checkpoint before figure generation

In [ ]:
# Cell 11.03
# Validate the frozen biological/statistical summary before plotting.
# Stop immediately if anything differs from the frozen analysis.

expected_traits = {
    "lrn",
    "scn_fi3",
    "prot_03",
    "days_fl_07"
}


observed_traits = set(
    four_qtl_full["trait"]
)


n_traits = len(
    all33_qtl
)

n_suggestive = int(
    (
        all33_qtl["genomewide_status"]
        == "suggestive_10pct"
    ).sum()
)

n_significant_05 = int(
    (
        all33_qtl["lod"]
        >=
        all33_qtl["lod_threshold_05pct"]
    ).sum()
)


validation = pd.DataFrame(
    [
        {
            "check": "33 traits present",
            "observed": n_traits,
            "expected": 33,
            "pass": n_traits == 33
        },
        {
            "check": "4 suggestive loci",
            "observed": n_suggestive,
            "expected": 4,
            "pass": n_suggestive == 4
        },
        {
            "check": "0 loci significant at 5%",
            "observed": n_significant_05,
            "expected": 0,
            "pass": n_significant_05 == 0
        },
        {
            "check": "Expected four suggestive traits",
            "observed": ", ".join(sorted(observed_traits)),
            "expected": ", ".join(sorted(expected_traits)),
            "pass": observed_traits == expected_traits
        }
    ]
)


print("NOTEBOOK 11 INPUT VALIDATION")
print("=" * 100)

display(validation)


if not validation["pass"].all():

    raise ValueError(
        "Frozen input validation failed. "
        "Do not generate manuscript figures until resolved."
    )


print()
print("All frozen-input checks passed.")

### Cell 11.04 — publication figure settings and save helper

In [ ]:
# Cell 11.04
# Define consistent publication settings and a reusable figure-saving helper.
#
# Every manuscript figure will automatically save:
#   1. PNG at 600 dpi
#   2. PDF
#
# Each plotting cell should also save its source data separately.

plt.rcParams.update(
    {
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 12,
        "axes.linewidth": 0.8,
        "lines.linewidth": 1.5,
        "lines.markersize": 5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.bbox": "tight"
    }
)


def save_manuscript_figure(
    fig,
    filename_stem,
    dpi=600
):
    """
    Save a manuscript figure as both high-resolution PNG and PDF.
    """

    png_path = (
        MANUSCRIPT_FIGURE_DIR
        / f"{filename_stem}.png"
    )

    pdf_path = (
        MANUSCRIPT_FIGURE_DIR
        / f"{filename_stem}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    print("SAVED")
    print(" PNG:", png_path)
    print(" PDF:", pdf_path)

    return png_path, pdf_path


def save_figure_source_table(
    df,
    filename_stem
):
    """
    Save the exact data used to generate a manuscript figure.
    """

    csv_path = (
        MANUSCRIPT_FIGURE_TABLE_DIR
        / f"{filename_stem}.csv"
    )

    df.to_csv(
        csv_path,
        index=False
    )

    print(" SOURCE DATA:", csv_path)

    return csv_path


print("Publication figure settings initialized.")

print()
print("Figure formats:")
print(" - PNG: 600 dpi")
print(" - PDF: vector")

print()
print("Figure outputs:")
print(MANUSCRIPT_FIGURE_DIR)

print()
print("Figure source tables:")
print(MANUSCRIPT_FIGURE_TABLE_DIR)

### Cell 11.05 — Figure 1: final structural linkage-map overview
* This shows the 22 final structural fragments, their provisional Kosambi lengths, marker counts when available, and chromosome assignments where supported.

In [ ]:
# Cell 11.05 — FINAL REVISED
# Figure 1 — Final structurally corrected linkage-map overview
#
# Improvements:
#   - chromosome assignment shown to right of each fragment
#   - marker count optionally shown inside/right of bar
#   - global map statistics added
#   - keeps provisional-cM wording explicit
#
# Saves:
#   PNG + PDF + source CSV

def find_first_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


# ----------------------------------------------------------
# Detect map-summary columns
# ----------------------------------------------------------

group_col = find_first_column(
    map_fragments,
    [
        "structural_group",
        "group",
        "linkage_group",
        "fragment",
        "group_name"
    ]
)

length_col = find_first_column(
    map_fragments,
    [
        "kosambi_length_provisional",
        "kosambi_cm_provisional",
        "total_kosambi_cm",
        "kosambi_length_cm",
        "map_length_kosambi",
        "length_kosambi_cm"
    ]
)

marker_col = find_first_column(
    map_fragments,
    [
        "n_markers",
        "marker_count",
        "n_framework_markers",
        "framework_markers"
    ]
)


if group_col is None:
    raise KeyError("Could not detect structural-group column.")

if length_col is None:
    raise KeyError("Could not detect provisional Kosambi length column.")


figure1_data = (
    map_fragments
    .copy()
    .rename(
        columns={
            group_col: "structural_group",
            length_col: "provisional_kosambi_cm"
        }
    )
)


if marker_col is not None:
    figure1_data = figure1_data.rename(
        columns={
            marker_col: "n_markers"
        }
    )


# ----------------------------------------------------------
# Frozen physical assignments
# Use explicit final assignments as fallback to guarantee
# stable labeling in the manuscript figure.
# ----------------------------------------------------------

final_chr_assignment = {
    "pLG01a_Gm06": "Gm06",
    "pLG01b_Gm20": "Gm20",
    "pLG02a": "Gm18",
    "pLG02b": "Gm18",
    "pLG03": "Gm17",
    "pLG04": "Gm02",
    "pLG05": "Gm01",
    "pLG06": "Gm08",
    "pLG07": "Gm08",
    "pLG08": "Gm03",
    "pLG09": "Gm13",
    "pLG10": "Gm09",
    "pLG11": "Gm20",
    "pLG12": "Gm11",
    "pLG13": "Gm12",
    "pLG14": "Gm16",
    "pLG15": "Gm06",
    "pLG16": "Gm15",
    "pLG17": "Gm07",
    "pLG18": "Gm13",
    "pLG19": "Gm08",
    "pLG20": "Gm07"
}


figure1_data[
    "physical_chr"
] = figure1_data[
    "structural_group"
].map(final_chr_assignment)


# ----------------------------------------------------------
# Sort longest → shortest visually
# ----------------------------------------------------------

figure1_data = (
    figure1_data
    .sort_values(
        "provisional_kosambi_cm",
        ascending=True
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Save exact plotting source
# ----------------------------------------------------------

save_figure_source_table(
    figure1_data,
    "Figure1_structural_map_overview_source"
)


# ----------------------------------------------------------
# Global frozen statistics
# ----------------------------------------------------------

n_fragments = len(figure1_data)

total_cm = (
    figure1_data[
        "provisional_kosambi_cm"
    ].sum()
)

n_framework = 166

n_removed_joins = 2


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9.3, 9.2)
)


y = np.arange(
    len(figure1_data)
)


bars = ax.barh(
    y,
    figure1_data[
        "provisional_kosambi_cm"
    ],
    height=0.70
)


ax.set_yticks(y)

ax.set_yticklabels(
    figure1_data[
        "structural_group"
    ],
    fontsize=8
)


# ----------------------------------------------------------
# Chromosome labels
# ----------------------------------------------------------

for bar, (_, row) in zip(
    bars,
    figure1_data.iterrows()
):

    bar_end = bar.get_width()

    chr_label = row[
        "physical_chr"
    ]

    if pd.notna(chr_label):

        ax.text(
            bar_end + 2.0,
            bar.get_y()
            + bar.get_height() / 2,
            chr_label,
            va="center",
            ha="left",
            fontsize=7.5
        )


# ----------------------------------------------------------
# Optional marker-count labels
# ----------------------------------------------------------

if "n_markers" in figure1_data.columns:

    for bar, (_, row) in zip(
        bars,
        figure1_data.iterrows()
    ):

        nmk = row[
            "n_markers"
        ]

        if pd.notna(nmk):

            ax.text(
                1.5,
                bar.get_y()
                + bar.get_height() / 2,
                f"{int(nmk)}",
                va="center",
                ha="left",
                fontsize=7,
                color="white",
                fontweight="bold"
            )


# ----------------------------------------------------------
# Axes
# ----------------------------------------------------------

ax.set_xlabel(
    "Provisional Kosambi map length (cM)"
)

ax.set_ylabel(
    "Structural linkage fragment"
)

ax.set_title(
    "Final structurally corrected Flyer × Hartwig linkage map"
)


ax.grid(
    axis="x",
    alpha=0.18
)


xmax = (
    figure1_data[
        "provisional_kosambi_cm"
    ].max()
)

ax.set_xlim(
    0,
    xmax * 1.22
)


# ----------------------------------------------------------
# Map summary note
# ----------------------------------------------------------

ax.text(
    0.985,
    0.02,
    (
        f"{n_fragments} structural fragments\n"
        f"{n_framework} ordered framework markers\n"
        f"{total_cm:.1f} provisional Kosambi cM\n"
        f"{n_removed_joins} unsupported joins removed"
    ),
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=7.8,
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        alpha=0.90,
        linewidth=0.5
    )
)


# ----------------------------------------------------------
# Small header for chromosome column
# ----------------------------------------------------------

ax.text(
    xmax * 1.07,
    len(figure1_data) - 0.15,
    "Physical\nchromosome",
    ha="center",
    va="bottom",
    fontsize=7.5,
    fontweight="bold"
)


plt.tight_layout()


# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

save_manuscript_figure(
    fig,
    "Figure1_structural_map_overview"
)

plt.show()
plt.close(fig)


print("FIGURE 1 SUMMARY")
print("=" * 100)

print(
    "Structural fragments:",
    n_fragments
)

print(
    "Total provisional Kosambi length:",
    round(total_cm, 6)
)

### Cell 11.06 — Figure 2: statistical support for the four suggestive loci
* This is more manuscript-friendly than a simple empirical-P bar chart because it shows each observed peak LOD relative to its own 10% and 5% empirical genome-wide thresholds.

In [ ]:
# Cell 11.06
# Final clean Figure 2:
# Empirical support for the four suggestive QTL

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Prepare the four frozen suggestive QTL
# ------------------------------------------------------------

fig2_df = four_qtl_full.copy()

required_cols = [
    "trait",
    "peak_marker",
    "lod",
    "lod_threshold_10pct",
    "lod_threshold_05pct",
    "peak_empirical_p",
]

missing = [c for c in required_cols if c not in fig2_df.columns]
if missing:
    raise KeyError(f"Missing required Figure 2 columns: {missing}")

# Preferred manuscript order
trait_order = [
    "scn_fi3",
    "prot_03",
    "days_fl_07",
    "lrn",
]

fig2_df["trait"] = pd.Categorical(
    fig2_df["trait"],
    categories=trait_order,
    ordered=True
)

fig2_df = (
    fig2_df
    .sort_values("trait")
    .reset_index(drop=True)
)

# Display labels
fig2_df["trait_marker"] = (
    fig2_df["trait"].astype(str)
    + "  |  "
    + fig2_df["peak_marker"].astype(str)
)

# y positions: first trait at top
y = np.arange(len(fig2_df))[::-1]

# ------------------------------------------------------------
# 2. Save exact source table
# ------------------------------------------------------------

fig2_source = fig2_df[
    [
        "trait",
        "peak_marker",
        "lod",
        "lod_threshold_10pct",
        "lod_threshold_05pct",
        "peak_empirical_p",
    ]
].copy()

save_figure_source_table(
    fig2_source,
    "Figure2_four_suggestive_qtl_thresholds_source"
)

# ------------------------------------------------------------
# 3. Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10.8, 5.8))

# Dotted connector between 10% and 5% thresholds
for yi, (_, row) in zip(y, fig2_df.iterrows()):

    x10 = float(row["lod_threshold_10pct"])
    x05 = float(row["lod_threshold_05pct"])
    obs = float(row["lod"])
    pval = float(row["peak_empirical_p"])

    # Light horizontal guide spanning the threshold interval
    ax.hlines(
        yi,
        xmin=min(x10, x05),
        xmax=max(x10, x05),
        linestyle=":",
        linewidth=1.2,
        alpha=0.50,
        zorder=1
    )

    # 10% empirical threshold
    ax.vlines(
        x10,
        yi - 0.115,
        yi + 0.115,
        linewidth=3.0,
        zorder=2
    )

    # 5% empirical threshold
    ax.vlines(
        x05,
        yi - 0.115,
        yi + 0.115,
        linewidth=3.0,
        zorder=2
    )

    # Observed peak LOD
    ax.scatter(
        obs,
        yi,
        s=105,
        zorder=4
    )

    # --------------------------------------------------------
    # Dynamic annotation placement:
    # always place text to the RIGHT of all three x positions
    # --------------------------------------------------------
    rightmost = max(obs, x10, x05)

    ax.text(
        rightmost + 0.045,
        yi,
        f"LOD {obs:.2f}; P={pval:.3f}",
        va="center",
        ha="left",
        fontsize=10.3,
        zorder=5
    )

# ------------------------------------------------------------
# 4. Axes and labels
# ------------------------------------------------------------

ax.set_yticks(y)
ax.set_yticklabels(
    fig2_df["trait_marker"],
    fontsize=10.5
)

ax.set_xlabel(
    "LOD score",
    fontsize=11.5
)

ax.set_ylabel(
    "Trait and peak marker",
    fontsize=11.5
)

ax.set_title(
    "Empirical support for the four suggestive QTL",
    fontsize=14,
    pad=30
)

# Dynamic x limits with enough room for annotations
xmin = min(
    fig2_df["lod"].min(),
    fig2_df["lod_threshold_10pct"].min(),
    fig2_df["lod_threshold_05pct"].min()
) - 0.22

xmax_data = max(
    fig2_df["lod"].max(),
    fig2_df["lod_threshold_10pct"].max(),
    fig2_df["lod_threshold_05pct"].max()
)

ax.set_xlim(
    xmin,
    xmax_data + 0.72
)

# Vertical grid only
ax.grid(
    axis="x",
    linestyle="-",
    linewidth=0.6,
    alpha=0.20
)

ax.grid(
    axis="y",
    visible=False
)

# ------------------------------------------------------------
# 5. Custom legend
# ------------------------------------------------------------

# Invisible artists purely for a clean legend
obs_handle = ax.scatter(
    [],
    [],
    s=90,
    label="Observed peak LOD"
)

thr10_handle = ax.plot(
    [],
    [],
    linestyle="None",
    marker="|",
    markersize=20,
    markeredgewidth=3,
    label="10% empirical threshold"
)[0]

thr05_handle = ax.plot(
    [],
    [],
    linestyle="None",
    marker="|",
    markersize=20,
    markeredgewidth=3,
    label="5% empirical threshold"
)[0]

ax.legend(
    handles=[
        obs_handle,
        thr10_handle,
        thr05_handle,
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 1.10),
    ncol=3,
    frameon=False,
    fontsize=10
)

# ------------------------------------------------------------
# 6. Interpretation note below plotting area
# ------------------------------------------------------------

fig.text(
    0.985,
    0.025,
    "All four loci exceeded the 10% genome-wide threshold; "
    "none reached the 5% threshold.",
    ha="right",
    va="bottom",
    fontsize=9.5
)

# Give labels/legend/note breathing room
fig.subplots_adjust(
    left=0.19,
    right=0.96,
    top=0.82,
    bottom=0.16
)

# ------------------------------------------------------------
# 7. Save PNG + PDF
# ------------------------------------------------------------

save_manuscript_figure(
    fig,
    "Figure2_four_suggestive_qtl_thresholds"
)

plt.show()

### Cell 11.07 — Figure 3: SCN FI3 regional QTL profile on Gm20
* This should be one of the most important biological figures because SCN has the best disease-related regional support.

In [ ]:
# Cell 11.07
# Figure 3 — SCN FI3 regional genetic profile
#
# Uses the frozen regional source table generated in Notebook 10.
#
# Saves revised manuscript source table + PNG + PDF.

SCN_REGION_FILE = (
    TABLE_DIR
    / "figure10_15_scn_fi3_regional_profile_source.csv"
)


if not SCN_REGION_FILE.exists():
    raise FileNotFoundError(
        f"SCN regional source table not found:\n{SCN_REGION_FILE}"
    )


scn_region = pd.read_csv(
    SCN_REGION_FILE
)


print("SCN REGION COLUMNS")
print("=" * 100)
print(scn_region.columns.tolist())


# ----------------------------------------------------------
# Detect provisional cM column
# ----------------------------------------------------------

cm_candidates = [
    "kosambi_cm_structural_provisional",
    "kosambi_cm_provisional",
    "kosambi_cm",
    "position_cm",
    "cm"
]

scn_cm_col = None

for col in cm_candidates:
    if col in scn_region.columns:
        scn_cm_col = col
        break


if scn_cm_col is None:
    raise KeyError(
        "Could not identify SCN genetic-position column."
    )


scn_region = (
    scn_region
    .sort_values(
        scn_cm_col
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Frozen threshold
# ----------------------------------------------------------

scn_master = (
    four_qtl_full
    .loc[
        four_qtl_full["trait"]
        == "scn_fi3"
    ]
    .iloc[0]
)


scn_threshold_10 = float(
    scn_master[
        "lod_threshold_10pct"
    ]
)


# ----------------------------------------------------------
# Save exact manuscript source data
# ----------------------------------------------------------

save_figure_source_table(
    scn_region,
    "Figure3_scn_fi3_gm20_regional_profile_source"
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.5, 5.6)
)


ax.plot(
    scn_region[scn_cm_col],
    scn_region["lod"],
    marker="o",
    linewidth=1.7
)


ax.axhline(
    scn_threshold_10,
    linestyle="--",
    linewidth=1.3,
    label=(
        "10% empirical genome-wide threshold "
        f"(LOD = {scn_threshold_10:.2f})"
    )
)


# ----------------------------------------------------------
# Highlight Satt354 and Satt270
# ----------------------------------------------------------

highlight_markers = [
    "Satt354",
    "Satt270"
]


for marker in highlight_markers:

    marker_row = (
        scn_region
        .loc[
            scn_region["marker"].astype(str)
            == marker
        ]
    )

    if len(marker_row) == 1:

        marker_row = marker_row.iloc[0]

        ax.scatter(
            marker_row[scn_cm_col],
            marker_row["lod"],
            s=70,
            zorder=5
        )

        ax.annotate(
            f"{marker}\nLOD={marker_row['lod']:.2f}",
            (
                marker_row[scn_cm_col],
                marker_row["lod"]
            ),
            xytext=(6, 8),
            textcoords="offset points",
            fontsize=8.5
        )


# ----------------------------------------------------------
# Label remaining markers lightly
# ----------------------------------------------------------

if len(scn_region) <= 12:

    for _, row in scn_region.iterrows():

        if str(row["marker"]) in highlight_markers:
            continue

        ax.annotate(
            str(row["marker"]),
            (
                row[scn_cm_col],
                row["lod"]
            ),
            xytext=(4, 3),
            textcoords="offset points",
            fontsize=7,
            alpha=0.75
        )


ax.set_xlabel(
    "Provisional Kosambi position (cM)"
)

ax.set_ylabel(
    "Single-marker LOD"
)

ax.set_title(
    "SCN FI3 QTL profile on pLG01b_Gm20"
)


ymax = max(
    scn_region["lod"].max(),
    scn_threshold_10
)

ax.set_ylim(
    0,
    ymax * 1.20
)


ax.grid(
    alpha=0.20
)


ax.legend(
    loc="upper left",
    frameon=False
)


plt.tight_layout()


save_manuscript_figure(
    fig,
    "Figure3_scn_fi3_gm20_regional_profile"
)

plt.show()
plt.close(fig)

### Cell 11.08 — Figure 4: allele effects across the SCN Gm20 region
* This complements Figure 3 by showing that the Flyer allele consistently reduces SCN female index in the local region.

In [ ]:
# Cell 11.08
# Figure 4 — SCN FI3 allele-effect profile across pLG01b_Gm20
#
# Negative effect_2_minus_0 means:
# Flyer allele (2) gives lower female index than Hartwig allele (0),
# i.e. greater SCN resistance.
#
# Saves source CSV + PNG + PDF.

required_cols = [
    "marker",
    scn_cm_col,
    "effect_2_minus_0",
    "lod"
]


missing_cols = [
    col
    for col in required_cols
    if col not in scn_region.columns
]


if missing_cols:
    raise KeyError(
        f"SCN regional table missing columns: {missing_cols}"
    )


figure4_data = (
    scn_region[
        required_cols
    ]
    .copy()
    .sort_values(
        scn_cm_col
    )
    .reset_index(drop=True)
)


figure4_data[
    "allele_direction"
] = np.where(
    figure4_data[
        "effect_2_minus_0"
    ] < 0,
    "Flyer allele lowers FI",
    "Hartwig allele lowers FI"
)


save_figure_source_table(
    figure4_data,
    "Figure4_scn_fi3_allele_effect_profile_source"
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.5, 5.4)
)


ax.axhline(
    0,
    linewidth=1.0
)


ax.plot(
    figure4_data[scn_cm_col],
    figure4_data[
        "effect_2_minus_0"
    ],
    marker="o",
    linewidth=1.7
)


for _, row in figure4_data.iterrows():

    ax.annotate(
        str(row["marker"]),
        (
            row[scn_cm_col],
            row["effect_2_minus_0"]
        ),
        xytext=(4, 5),
        textcoords="offset points",
        fontsize=7.5
    )


ax.set_xlabel(
    "Provisional Kosambi position (cM)"
)

ax.set_ylabel(
    "Flyer − Hartwig effect on SCN female index"
)

ax.set_title(
    "SCN FI3 allele effects across the Gm20 linkage fragment"
)


ax.text(
    0.02,
    0.04,
    "Negative values indicate lower female index\nwith the Flyer allele",
    transform=ax.transAxes,
    fontsize=8,
    va="bottom"
)


ax.grid(
    alpha=0.20
)


plt.tight_layout()


save_manuscript_figure(
    fig,
    "Figure4_scn_fi3_allele_effect_profile"
)

plt.show()
plt.close(fig)


print()
print("SCN EFFECT DIRECTION")
print("=" * 100)

print(
    figure4_data[
        "allele_direction"
    ]
    .value_counts()
)

display(
    figure4_data
)

### Cell 11.09 — Figure 5: SCN Gm20 physical bracket and anchor context
* This figure shows the defensible 28.303–38.482 Mb physical bracket, the anchor positions that define it, and the Satt354 QTL peak as genetically localized but physically unanchored.

In [ ]:
# Cell 11.09
# Figure 5 — SCN Gm20 physical bracket and anchor context
#
# Purpose:
#   Show the defensible anchor-defined physical bracket for the SCN FI3 QTL.
#
# IMPORTANT:
#   - This is NOT a statistical confidence interval.
#   - Satt354 has no direct Wm82.gnm6 coordinate.
#   - The physical bracket is defined by independent flanking anchors.
#
# Saves:
#   source CSV
#   PNG
#   PDF

SCN_CANDIDATE_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_scn_gm20_final_evidence_summary.xlsx"
)

if not SCN_CANDIDATE_FILE.exists():
    raise FileNotFoundError(
        f"SCN final evidence workbook not found:\n{SCN_CANDIDATE_FILE}"
    )


# ----------------------------------------------------------
# Frozen SCN physical bracket
# ----------------------------------------------------------

SCN_REGION_START_BP = 28_303_434
SCN_REGION_END_BP   = 38_482_498

SCN_REGION_START_MB = SCN_REGION_START_BP / 1e6
SCN_REGION_END_MB   = SCN_REGION_END_BP / 1e6


# ----------------------------------------------------------
# Load region summary if available
# ----------------------------------------------------------

scn_xls = pd.ExcelFile(
    SCN_CANDIDATE_FILE
)

print("SCN WORKBOOK SHEETS")
print("=" * 100)
print(scn_xls.sheet_names)


if "region_summary" in scn_xls.sheet_names:
    scn_region_summary = pd.read_excel(
        SCN_CANDIDATE_FILE,
        sheet_name="region_summary"
    )
else:
    scn_region_summary = pd.DataFrame()


# ----------------------------------------------------------
# Create explicit plotting table
# ----------------------------------------------------------

figure5_data = pd.DataFrame(
    [
        {
            "feature": "Left physical anchor",
            "physical_mb": SCN_REGION_START_MB,
            "feature_type": "anchor",
            "interpretation":
                "Defines left boundary of anchor-based bracket"
        },
        {
            "feature": "Right physical anchor",
            "physical_mb": SCN_REGION_END_MB,
            "feature_type": "anchor",
            "interpretation":
                "Defines right boundary of anchor-based bracket"
        }
    ]
)


FIGURE5_SOURCE = save_figure_source_table(
    figure5_data,
    "Figure5_scn_gm20_physical_bracket_source"
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9.0, 3.2)
)


# chromosome baseline
ax.hlines(
    y=0,
    xmin=SCN_REGION_START_MB - 1.5,
    xmax=SCN_REGION_END_MB + 1.5,
    linewidth=1.2
)


# physical bracket
ax.hlines(
    y=0,
    xmin=SCN_REGION_START_MB,
    xmax=SCN_REGION_END_MB,
    linewidth=8,
    alpha=0.35,
    label="Anchor-defined physical bracket"
)


# anchors
ax.scatter(
    [
        SCN_REGION_START_MB,
        SCN_REGION_END_MB
    ],
    [0, 0],
    s=90,
    zorder=5
)


ax.annotate(
    f"Left anchor\n{SCN_REGION_START_MB:.3f} Mb",
    (
        SCN_REGION_START_MB,
        0
    ),
    xytext=(0, 20),
    textcoords="offset points",
    ha="center",
    fontsize=9
)


ax.annotate(
    f"Right anchor\n{SCN_REGION_END_MB:.3f} Mb",
    (
        SCN_REGION_END_MB,
        0
    ),
    xytext=(0, 20),
    textcoords="offset points",
    ha="center",
    fontsize=9
)


# Satt354 annotation only — no invented coordinate
midpoint = (
    SCN_REGION_START_MB
    + SCN_REGION_END_MB
) / 2


ax.annotate(
    "QTL peak: Satt354\nno direct gnm6 coordinate",
    (
        midpoint,
        0
    ),
    xytext=(0, -42),
    textcoords="offset points",
    ha="center",
    fontsize=9
)


ax.set_xlim(
    SCN_REGION_START_MB - 1.5,
    SCN_REGION_END_MB + 1.5
)

ax.set_ylim(
    -0.55,
    0.55
)

ax.set_yticks([])

ax.set_xlabel(
    "Wm82.gnm6 physical position on Gm20 (Mb)"
)

ax.set_title(
    "SCN FI3 anchor-defined physical region on Gm20"
)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.22),
    frameon=False
)

ax.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()


save_manuscript_figure(
    fig,
    "Figure5_scn_gm20_physical_bracket"
)

plt.show()
plt.close(fig)


print()
print("SCN physical bracket")
print("=" * 100)

print(
    f"{SCN_REGION_START_MB:.6f}–"
    f"{SCN_REGION_END_MB:.6f} Mb"
)

print(
    "Span:",
    round(
        SCN_REGION_END_MB
        - SCN_REGION_START_MB,
        6
    ),
    "Mb"
)

print()
print(
    "IMPORTANT: anchor-defined physical bracket, "
    "not a statistical QTL confidence interval."
)

### Cell 11.10 — Figure 6: SCN high-priority candidate genes within the bracket
* This will plot the nine Tier-1 candidates and distinguish Glyma.20G104000 because it has direct SCN-responsive molecular evidence.

In [ ]:
# Cell 11.10 — MANUSCRIPT VERSION
# Figure 6 — High-priority SCN candidate genes across the Gm20 bracket
#
# Design:
#   - full gene names remain on the genomic figure
#   - labels are connected directly to their gene dots
#   - crowded distal candidates receive custom offsets
#   - Glyma.20G104000 is highlighted for direct SCN-responsive evidence
#   - no candidate-number key
#
# Saves:
#   - source CSV
#   - PNG
#   - PDF
#
# IMPORTANT:
# Candidate positions do NOT redefine or shrink the QTL bracket.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ----------------------------------------------------------
# Locate high-priority candidate sheet
# ----------------------------------------------------------

SCN_HIGH_SHEET = None

for candidate_sheet in [
    "high_priority_9",
    "high_priority",
    "Tier_1"
]:
    if candidate_sheet in scn_xls.sheet_names:
        SCN_HIGH_SHEET = candidate_sheet
        break


if SCN_HIGH_SHEET is None:
    raise KeyError(
        "Could not locate high-priority SCN candidate sheet. "
        f"Available sheets: {scn_xls.sheet_names}"
    )


scn_high = pd.read_excel(
    SCN_CANDIDATE_FILE,
    sheet_name=SCN_HIGH_SHEET
)


print("SCN HIGH-PRIORITY CANDIDATE COLUMNS")
print("=" * 100)
print(scn_high.columns.tolist())


# ----------------------------------------------------------
# Validate columns
# ----------------------------------------------------------

required_cols = [
    "gene",
    "start_mb_gnm6",
    "end_mb_gnm6"
]

missing_cols = [
    col
    for col in required_cols
    if col not in scn_high.columns
]

if missing_cols:
    raise KeyError(
        f"Missing required SCN columns: {missing_cols}"
    )


# ----------------------------------------------------------
# Prepare plotting data
# ----------------------------------------------------------

figure6_data = scn_high.copy()


figure6_data["start_mb_gnm6"] = pd.to_numeric(
    figure6_data["start_mb_gnm6"],
    errors="coerce"
)

figure6_data["end_mb_gnm6"] = pd.to_numeric(
    figure6_data["end_mb_gnm6"],
    errors="coerce"
)


# Gene midpoint — plotting only
figure6_data["plot_mb"] = (
    figure6_data["start_mb_gnm6"]
    + figure6_data["end_mb_gnm6"]
) / 2


figure6_data["gene_id"] = (
    figure6_data["gene"].astype(str)
)


# ----------------------------------------------------------
# Direct SCN evidence
# ----------------------------------------------------------

if "direct_SCN_evidence" in figure6_data.columns:

    figure6_data[
        "direct_scn_molecular_evidence"
    ] = (
        figure6_data["direct_SCN_evidence"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "yes",
                "1",
                "direct",
                "supported"
            ]
        )
    )

else:

    figure6_data[
        "direct_scn_molecular_evidence"
    ] = (
        figure6_data["gene_id"]
        == "Glyma.20G104000"
    )


# Frozen-result safety check
figure6_data.loc[
    figure6_data["gene_id"]
    == "Glyma.20G104000",
    "direct_scn_molecular_evidence"
] = True


figure6_data = (
    figure6_data
    .dropna(
        subset=["plot_mb"]
    )
    .sort_values(
        "plot_mb"
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Vertical tracks for the gene dots
#
# Purely graphical — no biological meaning.
# ----------------------------------------------------------

dot_tracks = {

    "Glyma.20G078700":  0.55,
    "Glyma.20G078800": -0.48,

    "Glyma.20G080700":  0.72,
    "Glyma.20G083000": -0.68,

    "Glyma.20G100000":  0.52,
    "Glyma.20G100500": -0.46,
    "Glyma.20G101100":  0.77,
    "Glyma.20G104000": -0.78,

    "Glyma.20G111300":  0.58
}


figure6_data["plot_track"] = (
    figure6_data["gene_id"]
    .map(dot_tracks)
    .fillna(0.45)
)


# ----------------------------------------------------------
# Custom label offsets
#
# Units = display points from each gene dot.
# These are manually spaced to avoid overlap.
# ----------------------------------------------------------

label_offsets = {

    "Glyma.20G078700": (  8,  10),
    "Glyma.20G078800": (  8, -12),

    "Glyma.20G080700": (  8,  12),
    "Glyma.20G083000": (  8, -12),

    # Dense distal cluster
    "Glyma.20G100000": (-10,  16),
    "Glyma.20G100500": (-18, -18),
    "Glyma.20G101100": ( 10,  22),

    # Special direct-evidence candidate
    "Glyma.20G104000": (-34, -32),

    # Pull inward from right edge
    "Glyma.20G111300": (-10,  12)
}


# ----------------------------------------------------------
# Horizontal alignment for each label
# ----------------------------------------------------------

label_alignment = {

    "Glyma.20G078700": "left",
    "Glyma.20G078800": "left",

    "Glyma.20G080700": "left",
    "Glyma.20G083000": "left",

    "Glyma.20G100000": "right",
    "Glyma.20G100500": "right",
    "Glyma.20G101100": "left",
    "Glyma.20G104000": "right",
    "Glyma.20G111300": "right"
}


# ----------------------------------------------------------
# Save source table
# ----------------------------------------------------------

save_figure_source_table(
    figure6_data,
    "Figure6_scn_gm20_high_priority_candidates_source"
)


# ----------------------------------------------------------
# Create figure
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(12.0, 6.8)
)


# ----------------------------------------------------------
# Anchor-defined physical bracket
# ----------------------------------------------------------

ax.axvspan(
    SCN_REGION_START_MB,
    SCN_REGION_END_MB,
    alpha=0.08,
    label="Anchor-defined SCN region"
)


ax.axvline(
    SCN_REGION_START_MB,
    linestyle=":",
    linewidth=1.0
)

ax.axvline(
    SCN_REGION_END_MB,
    linestyle=":",
    linewidth=1.0
)


# Central genomic track
ax.hlines(
    y=0,
    xmin=SCN_REGION_START_MB,
    xmax=SCN_REGION_END_MB,
    linewidth=1.3
)


# ----------------------------------------------------------
# Draw candidate genes
# ----------------------------------------------------------

for _, row in figure6_data.iterrows():

    gene = row["gene_id"]
    x = row["plot_mb"]
    y = row["plot_track"]


    # Connector from genomic baseline to gene dot
    ax.vlines(
        x,
        0,
        y,
        linewidth=0.9
    )


    # Larger point for direct SCN molecular evidence
    marker_size = (
        100
        if row["direct_scn_molecular_evidence"]
        else 55
    )


    ax.scatter(
        x,
        y,
        s=marker_size,
        zorder=5
    )


    # ------------------------------------------------------
    # Label settings
    # ------------------------------------------------------

    dx, dy = label_offsets.get(
        gene,
        (6, 8)
    )

    ha = label_alignment.get(
        gene,
        "left"
    )


    # ------------------------------------------------------
    # Special text for Glyma.20G104000
    # ------------------------------------------------------

    if gene == "Glyma.20G104000":

        label_text = (
            "Glyma.20G104000\n"
            "SCN-responsive evidence"
        )

        fontweight = "bold"

    else:

        label_text = gene
        fontweight = "normal"


    # ------------------------------------------------------
    # Gene annotation with leader line
    # ------------------------------------------------------

    ax.annotate(
        label_text,

        xy=(
            x,
            y
        ),

        xytext=(
            dx,
            dy
        ),

        textcoords="offset points",

        ha=ha,

        va=(
            "bottom"
            if dy >= 0
            else "top"
        ),

        fontsize=7.7,

        fontweight=fontweight,

        rotation=0,

        arrowprops=dict(
            arrowstyle="-",
            linewidth=0.6,
            shrinkA=2,
            shrinkB=2
        ),

        # only give direct-evidence candidate
        # a subtle white background
        bbox=(
            dict(
                boxstyle="round,pad=0.15",
                facecolor="white",
                alpha=0.82,
                linewidth=0
            )
            if gene == "Glyma.20G104000"
            else None
        )
    )


# ----------------------------------------------------------
# Bracket-coordinate labels
# ----------------------------------------------------------

ax.text(
    SCN_REGION_START_MB,
    1.00,
    f"{SCN_REGION_START_MB:.3f} Mb",
    ha="center",
    va="bottom",
    fontsize=8
)


ax.text(
    SCN_REGION_END_MB,
    1.00,
    f"{SCN_REGION_END_MB:.3f} Mb",
    ha="center",
    va="bottom",
    fontsize=8
)


# ----------------------------------------------------------
# Axes
# ----------------------------------------------------------

ax.set_xlim(
    SCN_REGION_START_MB - 0.45,
    SCN_REGION_END_MB + 0.65
)


ax.set_ylim(
    -1.30,
    1.30
)


ax.set_yticks([])


ax.set_xlabel(
    "Wm82.gnm6 physical position on Gm20 (Mb)"
)


ax.set_title(
    "High-priority candidate genes within the SCN Gm20 region"
)


ax.grid(
    axis="x",
    alpha=0.18
)


ax.legend(
    loc="upper left",
    frameon=False
)


# ----------------------------------------------------------
# Figure note
# ----------------------------------------------------------

ax.text(
    0.01,
    0.025,
    (
        "Candidate positions are shown within the full "
        "anchor-defined physical region; candidate density "
        "was not used to narrow the QTL."
    ),
    transform=ax.transAxes,
    fontsize=7.8,
    ha="left",
    va="bottom"
)


fig.subplots_adjust(
    top=0.90,
    bottom=0.15,
    left=0.07,
    right=0.98
)


# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

save_manuscript_figure(
    fig,
    "Figure6_scn_gm20_high_priority_candidates"
)


plt.show()
plt.close(fig)


# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()
print("SCN HIGH-PRIORITY CANDIDATE FIGURE")
print("=" * 100)

print(
    "Candidate genes plotted:",
    len(figure6_data)
)

print(
    "Direct SCN molecular evidence:",
    int(
        figure6_data[
            "direct_scn_molecular_evidence"
        ].sum()
    )
)


print()

display(
    figure6_data[
        [
            "gene_id",
            "start_mb_gnm6",
            "end_mb_gnm6",
            "plot_mb",
            "plot_track",
            "direct_scn_molecular_evidence"
        ]
    ]
)

### Cell 11.11 — Figure S1: LRN Gm02 physical bracket and candidate shortlist
* This uses the full 20.165739–46.904614 Mb anchor-defined bracket and plots the 8 evidence-aware candidates without pretending the candidate cluster narrows the QTL.

In [ ]:
# Cell 11.11
# Figure S1 — LRN Gm02 physical bracket and evidence-aware candidate shortlist
#
# Saves:
#   - source CSV
#   - PNG
#   - PDF
#
# IMPORTANT:
# The full anchor-defined bracket is retained.
# Candidate positions do NOT narrow the QTL interval.

LRN_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_lrn_gm02_final_candidates.xlsx"
)

if not LRN_FILE.exists():
    raise FileNotFoundError(
        f"LRN final candidate workbook not found:\n{LRN_FILE}"
    )


lrn_xls = pd.ExcelFile(
    LRN_FILE
)


print("LRN WORKBOOK SHEETS")
print("=" * 100)
print(lrn_xls.sheet_names)


# ----------------------------------------------------------
# Locate evidence shortlist
# ----------------------------------------------------------

LRN_SHEET = None

for candidate_sheet in [
    "evidence_shortlist",
    "shortlist"
]:
    if candidate_sheet in lrn_xls.sheet_names:
        LRN_SHEET = candidate_sheet
        break


if LRN_SHEET is None:
    raise KeyError(
        "Could not locate LRN evidence shortlist."
    )


lrn_candidates = pd.read_excel(
    LRN_FILE,
    sheet_name=LRN_SHEET
)


print()
print("LRN CANDIDATE COLUMNS")
print("=" * 100)
print(lrn_candidates.columns.tolist())


# ----------------------------------------------------------
# Helper
# ----------------------------------------------------------

def first_existing(df, candidates):

    for col in candidates:
        if col in df.columns:
            return col

    return None


gene_col = first_existing(
    lrn_candidates,
    [
        "gene",
        "gene_id",
        "gene_model",
        "Name",
        "ID"
    ]
)


start_mb_col = first_existing(
    lrn_candidates,
    [
        "start_mb_gnm6",
        "start_mb",
        "gene_start_mb"
    ]
)


end_mb_col = first_existing(
    lrn_candidates,
    [
        "end_mb_gnm6",
        "end_mb",
        "gene_end_mb"
    ]
)


mid_mb_col = first_existing(
    lrn_candidates,
    [
        "position_mb",
        "midpoint_mb",
        "physical_mb",
        "gene_midpoint_mb"
    ]
)


if gene_col is None:
    raise KeyError(
        "Could not identify LRN gene ID column."
    )


# ----------------------------------------------------------
# Prepare plotting data
# ----------------------------------------------------------

figureS1_data = (
    lrn_candidates
    .copy()
)


figureS1_data[
    "gene_id"
] = figureS1_data[
    gene_col
].astype(str)


if (
    start_mb_col is not None
    and end_mb_col is not None
):

    figureS1_data[
        "plot_mb"
    ] = (
        pd.to_numeric(
            figureS1_data[start_mb_col],
            errors="coerce"
        )
        +
        pd.to_numeric(
            figureS1_data[end_mb_col],
            errors="coerce"
        )
    ) / 2

elif mid_mb_col is not None:

    figureS1_data[
        "plot_mb"
    ] = pd.to_numeric(
        figureS1_data[mid_mb_col],
        errors="coerce"
    )

else:

    raise KeyError(
        "Could not identify LRN physical-position columns."
    )


figureS1_data = (
    figureS1_data
    .dropna(
        subset=["plot_mb"]
    )
    .sort_values(
        "plot_mb"
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Frozen physical context
# ----------------------------------------------------------

LRN_START_MB = 20.165739
LRN_END_MB = 46.904614
LRN_SATT282A_ANCHOR_MB = 30.354248


# ----------------------------------------------------------
# Graphical tracks
# ----------------------------------------------------------

tracks = np.where(
    np.arange(
        len(figureS1_data)
    ) % 2 == 0,
    0.50,
    -0.50
)


figureS1_data[
    "plot_track"
] = tracks


# ----------------------------------------------------------
# Save source table
# ----------------------------------------------------------

save_figure_source_table(
    figureS1_data,
    "FigureS1_lrn_gm02_candidate_region_source"
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11.5, 6.5)
)


ax.axvspan(
    LRN_START_MB,
    LRN_END_MB,
    alpha=0.08,
    label="Anchor-defined LRN region"
)


ax.axvline(
    LRN_START_MB,
    linestyle=":",
    linewidth=1.0
)

ax.axvline(
    LRN_END_MB,
    linestyle=":",
    linewidth=1.0
)


# Satt282a family anchor
ax.axvline(
    LRN_SATT282A_ANCHOR_MB,
    linestyle="--",
    linewidth=1.2,
    label="Satt282a assay-family anchor"
)


# physical baseline
ax.hlines(
    y=0,
    xmin=LRN_START_MB,
    xmax=LRN_END_MB,
    linewidth=1.2
)


# ----------------------------------------------------------
# Candidate genes
# ----------------------------------------------------------

for _, row in figureS1_data.iterrows():

    x = row["plot_mb"]
    y = row["plot_track"]

    ax.vlines(
        x,
        0,
        y,
        linewidth=0.8
    )

    ax.scatter(
        x,
        y,
        s=55,
        zorder=5
    )

    ax.annotate(
        row["gene_id"],
        (x, y),
        xytext=(
            5,
            7 if y > 0 else -7
        ),
        textcoords="offset points",
        ha="left",
        va=(
            "bottom"
            if y > 0
            else "top"
        ),
        fontsize=7.5,
        rotation=25,
        arrowprops=dict(
            arrowstyle="-",
            linewidth=0.5
        )
    )


# ----------------------------------------------------------
# Anchor label
# ----------------------------------------------------------

ax.annotate(
    "Satt282a family anchor\n30.354 Mb",
    (
        LRN_SATT282A_ANCHOR_MB,
        0
    ),
    xytext=(8, 26),
    textcoords="offset points",
    fontsize=8
)


# ----------------------------------------------------------
# Axes
# ----------------------------------------------------------

ax.set_xlim(
    LRN_START_MB - 1.0,
    LRN_END_MB + 1.0
)

ax.set_ylim(
    -1.15,
    1.15
)

ax.set_yticks([])

ax.set_xlabel(
    "Wm82.gnm6 physical position on Gm02 (Mb)"
)

ax.set_title(
    "LRN QTL physical region and evidence-aware candidate shortlist"
)

ax.grid(
    axis="x",
    alpha=0.18
)

ax.legend(
    loc="upper left",
    frameon=False
)


ax.text(
    0.01,
    0.025,
    (
        "The full anchor-defined bracket is retained; "
        "candidate locations were not used to narrow the QTL."
    ),
    transform=ax.transAxes,
    fontsize=7.8
)


fig.subplots_adjust(
    top=0.90,
    bottom=0.15,
    left=0.07,
    right=0.98
)


save_manuscript_figure(
    fig,
    "FigureS1_lrn_gm02_candidate_region"
)

plt.show()
plt.close(fig)


print()
print("LRN FIGURE SUMMARY")
print("=" * 100)

print(
    "Candidates plotted:",
    len(figureS1_data)
)

print(
    "Physical bracket:",
    f"{LRN_START_MB:.6f}–{LRN_END_MB:.6f} Mb"
)

print(
    "Bracket span:",
    round(
        LRN_END_MB - LRN_START_MB,
        6
    ),
    "Mb"
)

display(
    figureS1_data[
        [
            "gene_id",
            "plot_mb"
        ]
    ]
)

### Cell 11.12 — Figure S2: prot_03 Satt440 physical-anchor context
* Here we deliberately do not imply a QTL interval. Satt440 is an exact physical peak anchor at 49.904048 Mb, and the ±1 Mb view is purely descriptive.

In [ ]:
# Cell 11.12
# Figure S2 — prot_03 anchor-centered physical context on Gm20
#
# IMPORTANT:
#   - Satt440 has an exact physical coordinate.
#   - There is NO closed physical QTL interval.
#   - ±1 Mb is shown only as descriptive anchor-centered context.
#
# Saves source CSV + PNG + PDF.

PROT03_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_prot03_gm20_final_evidence_summary.xlsx"
)

if not PROT03_FILE.exists():
    raise FileNotFoundError(
        f"prot_03 workbook not found:\n{PROT03_FILE}"
    )


prot_xls = pd.ExcelFile(
    PROT03_FILE
)


print("prot_03 WORKBOOK SHEETS")
print("=" * 100)
print(prot_xls.sheet_names)


# ----------------------------------------------------------
# Load curated candidates
# ----------------------------------------------------------

PROT_SHEET = None

for candidate_sheet in [
    "curated_candidates",
    "focused_candidates"
]:
    if candidate_sheet in prot_xls.sheet_names:
        PROT_SHEET = candidate_sheet
        break


if PROT_SHEET is None:
    raise KeyError(
        "Could not locate prot_03 curated candidate sheet."
    )


prot_candidates = pd.read_excel(
    PROT03_FILE,
    sheet_name=PROT_SHEET
)


print()
print("prot_03 CANDIDATE COLUMNS")
print("=" * 100)
print(prot_candidates.columns.tolist())


# ----------------------------------------------------------
# Detect columns
# ----------------------------------------------------------

gene_col = first_existing(
    prot_candidates,
    [
        "gene",
        "gene_id",
        "gene_model",
        "Name",
        "ID"
    ]
)


start_mb_col = first_existing(
    prot_candidates,
    [
        "start_mb_gnm6",
        "start_mb",
        "gene_start_mb"
    ]
)


end_mb_col = first_existing(
    prot_candidates,
    [
        "end_mb_gnm6",
        "end_mb",
        "gene_end_mb"
    ]
)


mid_mb_col = first_existing(
    prot_candidates,
    [
        "position_mb",
        "midpoint_mb",
        "physical_mb",
        "gene_midpoint_mb"
    ]
)


if gene_col is None:
    raise KeyError(
        "Could not identify prot_03 gene column."
    )


# ----------------------------------------------------------
# Prepare plotting data
# ----------------------------------------------------------

figureS2_data = prot_candidates.copy()


figureS2_data[
    "gene_id"
] = figureS2_data[
    gene_col
].astype(str)


if (
    start_mb_col is not None
    and end_mb_col is not None
):

    figureS2_data[
        "plot_mb"
    ] = (
        pd.to_numeric(
            figureS2_data[start_mb_col],
            errors="coerce"
        )
        +
        pd.to_numeric(
            figureS2_data[end_mb_col],
            errors="coerce"
        )
    ) / 2

elif mid_mb_col is not None:

    figureS2_data[
        "plot_mb"
    ] = pd.to_numeric(
        figureS2_data[mid_mb_col],
        errors="coerce"
    )

else:

    raise KeyError(
        "Could not identify prot_03 position columns."
    )


figureS2_data = (
    figureS2_data
    .dropna(
        subset=["plot_mb"]
    )
    .sort_values(
        "plot_mb"
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Frozen Satt440 coordinate
# ----------------------------------------------------------

SATT440_MB = 49.904048

WINDOW_START = SATT440_MB - 1.0
WINDOW_END = SATT440_MB + 1.0


# retain only descriptive ±1 Mb context
figureS2_plot = (
    figureS2_data
    .loc[
        figureS2_data["plot_mb"]
        .between(
            WINDOW_START,
            WINDOW_END
        )
    ]
    .copy()
    .reset_index(drop=True)
)


save_figure_source_table(
    figureS2_plot,
    "FigureS2_prot03_gm20_anchor_context_source"
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11.0, 6.0)
)


# descriptive context window
ax.axvspan(
    WINDOW_START,
    WINDOW_END,
    alpha=0.07,
    label="Descriptive ±1 Mb context"
)


# exact peak anchor
ax.axvline(
    SATT440_MB,
    linestyle="--",
    linewidth=1.4,
    label="Satt440 exact anchor"
)


ax.hlines(
    y=0,
    xmin=WINDOW_START,
    xmax=WINDOW_END,
    linewidth=1.1
)


# ----------------------------------------------------------
# Plot curated candidates
# ----------------------------------------------------------

for i, row in figureS2_plot.iterrows():

    y = (
        0.50
        if i % 2 == 0
        else -0.50
    )

    ax.vlines(
        row["plot_mb"],
        0,
        y,
        linewidth=0.7
    )

    ax.scatter(
        row["plot_mb"],
        y,
        s=38
    )


    # Label only the strongest candidates directly
    if row["gene_id"] in [
        "Glyma.20G228900",
        "Glyma.20G229000",
        "Glyma.20G235500"
    ]:

        ax.annotate(
            row["gene_id"],
            (
                row["plot_mb"],
                y
            ),
            xytext=(
                5,
                7 if y > 0 else -7
            ),
            textcoords="offset points",
            fontsize=8,
            ha="left",
            va=(
                "bottom"
                if y > 0
                else "top"
            )
        )


# ----------------------------------------------------------
# Satt440 label
# ----------------------------------------------------------

ax.annotate(
    "Satt440\n49.904 Mb",
    (
        SATT440_MB,
        0
    ),
    xytext=(8, 28),
    textcoords="offset points",
    fontsize=8.5
)


ax.set_xlim(
    WINDOW_START,
    WINDOW_END
)

ax.set_ylim(
    -1.15,
    1.15
)

ax.set_yticks([])

ax.set_xlabel(
    "Wm82.gnm6 physical position on Gm20 (Mb)"
)

ax.set_title(
    "prot_03 anchor-centered physical context around Satt440"
)

ax.legend(
    loc="upper left",
    frameon=False
)

ax.grid(
    axis="x",
    alpha=0.18
)


ax.text(
    0.01,
    0.025,
    (
        "The ±1 Mb span is descriptive context only; "
        "it is not a QTL confidence interval."
    ),
    transform=ax.transAxes,
    fontsize=7.8
)


fig.subplots_adjust(
    top=0.90,
    bottom=0.15,
    left=0.07,
    right=0.98
)


save_manuscript_figure(
    fig,
    "FigureS2_prot03_gm20_anchor_context"
)

plt.show()
plt.close(fig)


print()
print("prot_03 FIGURE SUMMARY")
print("=" * 100)

print(
    "Curated candidates in ±1 Mb context:",
    len(figureS2_plot)
)

print(
    "Exact peak anchor:",
    f"{SATT440_MB:.6f} Mb"
)

### Cell 11.13 — Figure S3: days_fl_07 single-anchor physical context
* This figure makes the limitation visually explicit: TMA2 is not physically anchored, while Sat_162 is the sole exact Gm08 anchor.

In [ ]:
# Cell 11.13
# Figure S3 — days_fl_07 single-anchor physical context
#
# No candidate-gene window is inferred.
#
# Saves source CSV + PNG + PDF.

DAYSFL_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_days_fl07_gm08_context.xlsx"
)


if not DAYSFL_FILE.exists():
    raise FileNotFoundError(
        f"days_fl_07 context workbook not found:\n{DAYSFL_FILE}"
    )


days_xls = pd.ExcelFile(
    DAYSFL_FILE
)


print("days_fl_07 WORKBOOK SHEETS")
print("=" * 100)
print(days_xls.sheet_names)


# ----------------------------------------------------------
# Frozen values
# ----------------------------------------------------------

TMA2_CM = 16.252318
SAT162_CM = 18.994791

SAT162_MB = 8.327526

GENETIC_SEPARATION_CM = (
    SAT162_CM - TMA2_CM
)


figureS3_data = pd.DataFrame(
    [
        {
            "marker": "TMA2",
            "provisional_cm": TMA2_CM,
            "physical_mb": np.nan,
            "physical_status":
                "No direct physical coordinate"
        },
        {
            "marker": "Sat_162",
            "provisional_cm": SAT162_CM,
            "physical_mb": SAT162_MB,
            "physical_status":
                "Exact Gm08 anchor"
        }
    ]
)


save_figure_source_table(
    figureS3_data,
    "FigureS3_days_fl07_single_anchor_context_source"
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.5, 4.8)
)


# genetic track
ax.hlines(
    y=0,
    xmin=TMA2_CM - 1,
    xmax=SAT162_CM + 1,
    linewidth=1.4
)


ax.scatter(
    [
        TMA2_CM,
        SAT162_CM
    ],
    [
        0,
        0
    ],
    s=[
        75,
        75
    ],
    zorder=5
)


ax.annotate(
    "TMA2\nQTL peak\nno direct physical coordinate",
    (
        TMA2_CM,
        0
    ),
    xytext=(-10, 32),
    textcoords="offset points",
    ha="center",
    fontsize=8.5
)


ax.annotate(
    "Sat_162\nexact Gm08 anchor\n8.328 Mb",
    (
        SAT162_CM,
        0
    ),
    xytext=(10, 32),
    textcoords="offset points",
    ha="center",
    fontsize=8.5
)


# genetic separation
mid_cm = (
    TMA2_CM + SAT162_CM
) / 2


ax.annotate(
    f"{GENETIC_SEPARATION_CM:.3f} provisional cM",
    (
        mid_cm,
        0
    ),
    xytext=(0, -30),
    textcoords="offset points",
    ha="center",
    fontsize=8
)


ax.set_xlim(
    TMA2_CM - 1.2,
    SAT162_CM + 1.2
)

ax.set_ylim(
    -0.65,
    0.65
)

ax.set_yticks([])

ax.set_xlabel(
    "Provisional Kosambi position on pLG07 (cM)"
)

ax.set_title(
    "days_fl_07 QTL: single-anchor physical assignment to Gm08"
)


ax.text(
    0.5,
    0.05,
    (
        "A single physical anchor cannot define the physical "
        "position or Mb interval of TMA2."
    ),
    transform=ax.transAxes,
    ha="center",
    fontsize=8
)


ax.grid(
    axis="x",
    alpha=0.18
)


plt.tight_layout()


save_manuscript_figure(
    fig,
    "FigureS3_days_fl07_single_anchor_context"
)

plt.show()
plt.close(fig)


print()
print("days_fl_07 PHYSICAL CONTEXT")
print("=" * 100)

print(
    "TMA2 → Sat_162:",
    round(
        GENETIC_SEPARATION_CM,
        6
    ),
    "provisional cM"
)

print(
    "Sat_162 exact physical coordinate:",
    SAT162_MB,
    "Mb"
)

### Cell 11.14 — Figure S4: physical-resolution hierarchy across the four suggestive QTL
* This summarizes why SCN/LRN got bracket-level candidate analyses, prot_03 got anchor-centered context, and days_fl_07 did not get candidate mining.

In [ ]:
# Cell 11.14 — REVISED
# Figure S4 — Physical-resolution hierarchy of the four suggestive QTL
#
# This is a categorical evidence visualization.
# It is NOT a statistical score.
#
# Saves source CSV + PNG + PDF.

figureS4_data = pd.DataFrame(
    [
        {
            "trait": "scn_fi3",
            "peak_marker": "Satt354",
            "chromosome": "Gm20",
            "physical_resolution":
                "Two-sided anchor bracket",
            "resolution_level": 3,
            "candidate_analysis":
                "Prioritized candidate region"
        },
        {
            "trait": "lrn",
            "peak_marker": "Satt282a",
            "chromosome": "Gm02",
            "physical_resolution":
                "Two-sided anchor bracket",
            "resolution_level": 3,
            "candidate_analysis":
                "Evidence-aware shortlist"
        },
        {
            "trait": "prot_03",
            "peak_marker": "Satt440",
            "chromosome": "Gm20",
            "physical_resolution":
                "Exact peak anchor only",
            "resolution_level": 2,
            "candidate_analysis":
                "Descriptive local context"
        },
        {
            "trait": "days_fl_07",
            "peak_marker": "TMA2",
            "chromosome": "Gm08",
            "physical_resolution":
                "Single-anchor group assignment",
            "resolution_level": 1,
            "candidate_analysis":
                "No candidate window"
        }
    ]
)


figureS4_data[
    "plot_label"
] = (
    figureS4_data["trait"]
    + "\n"
    + figureS4_data["peak_marker"]
)


# ----------------------------------------------------------
# Save source data
# ----------------------------------------------------------

save_figure_source_table(
    figureS4_data,
    "FigureS4_qtl_physical_resolution_hierarchy_source"
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.8, 5.8)
)


x = np.arange(
    len(figureS4_data)
)


bars = ax.bar(
    x,
    figureS4_data[
        "resolution_level"
    ],
    width=0.62
)


# ----------------------------------------------------------
# X axis
# ----------------------------------------------------------

ax.set_xticks(x)

ax.set_xticklabels(
    figureS4_data[
        "plot_label"
    ],
    fontsize=9
)


# ----------------------------------------------------------
# Y axis
# ----------------------------------------------------------

ax.set_yticks(
    [
        1,
        2,
        3
    ]
)


ax.set_yticklabels(
    [
        "Single-anchor\ngroup assignment",
        "Exact peak\nanchor only",
        "Two-sided\nanchor bracket"
    ]
)


ax.set_ylim(
    0,
    3.65
)


ax.set_ylabel(
    "Physical localization evidence"
)

ax.set_xlabel(
    "Trait and peak marker"
)


ax.set_title(
    "Physical-resolution hierarchy of the four suggestive QTL"
)


# ----------------------------------------------------------
# Candidate-analysis labels above bars
# ----------------------------------------------------------

for bar, (_, row) in zip(
    bars,
    figureS4_data.iterrows()
):

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height() + 0.10,
        row["candidate_analysis"],
        ha="center",
        va="bottom",
        fontsize=7.5
    )


# ----------------------------------------------------------
# Grid
# ----------------------------------------------------------

ax.grid(
    axis="y",
    alpha=0.18
)


# ----------------------------------------------------------
# Explanatory note — TOP RIGHT
# ----------------------------------------------------------

ax.text(
    0.985,
    0.965,
    (
        "Levels represent physical-resolution categories,\n"
        "not statistical significance or QTL strength."
    ),
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=7.8,
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        alpha=0.85,
        linewidth=0.5
    )
)


# ----------------------------------------------------------
# Layout
# ----------------------------------------------------------

fig.subplots_adjust(
    top=0.88,
    bottom=0.18,
    left=0.18,
    right=0.98
)


# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

save_manuscript_figure(
    fig,
    "FigureS4_qtl_physical_resolution_hierarchy"
)

plt.show()
plt.close(fig)


# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print("PHYSICAL-RESOLUTION SUMMARY")
print("=" * 100)

display(
    figureS4_data[
        [
            "trait",
            "peak_marker",
            "chromosome",
            "physical_resolution",
            "candidate_analysis"
        ]
    ]
)

### Cell 11.15 — load frozen disease QTL results

In [ ]:
# Cell 11.15
# Load frozen disease-trait QTL results for manuscript disease figures.
#
# No QTL analysis is rerun here.

QTL_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_all33_empirical_qtl_screen.xlsx"
)


if not QTL_FILE.exists():
    raise FileNotFoundError(
        f"Frozen QTL workbook not found:\n{QTL_FILE}"
    )


disease_traits = [
    "scn_fi3",
    "scn_fi14",
    "sds_di",
    "sds_ds",
    "sds_dx",
    "sds_dx_mean"
]


disease_qtl = (
    all33_qtl
    .loc[
        all33_qtl["trait"].isin(
            disease_traits
        )
    ]
    .copy()
)


# explicit desired order
trait_order = {
    "scn_fi3": 1,
    "scn_fi14": 2,
    "sds_di": 3,
    "sds_ds": 4,
    "sds_dx": 5,
    "sds_dx_mean": 6
}


disease_qtl[
    "_order"
] = disease_qtl[
    "trait"
].map(trait_order)


disease_qtl = (
    disease_qtl
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


save_figure_source_table(
    disease_qtl,
    "Figure7_disease_qtl_summary_source"
)


print("FROZEN DISEASE QTL RESULTS")
print("=" * 100)

display(
    disease_qtl[
        [
            "trait",
            "peak_marker",
            "candidate_chr",
            "n",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
)

### Cell 11.16 — Figure 7: disease QTL evidence across SCN and SDS traits
* This should be useful in the main manuscript because it prevents overstatement of SDS.

In [ ]:
# Cell 11.16 — REVISED
# Figure 7 — Disease QTL evidence across SCN and SDS phenotypes
#
# Improvements:
#   - observed LOD labels moved below points
#   - avoids overlap with 10% / 5% threshold markers
#   - cleaner legend/title spacing
#   - compact interpretation note at upper right
#   - saves PNG + PDF
#
# Scientific message:
#   Only SCN FI3 exceeds its trait-specific 10% empirical threshold.
#   No disease trait reaches its 5% empirical threshold.

figure7_data = disease_qtl.copy()


figure7_data["label"] = (
    figure7_data["trait"].astype(str)
    + "\n"
    + figure7_data["peak_marker"].astype(str)
)


x = np.arange(
    len(figure7_data)
)


# ----------------------------------------------------------
# Figure
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10.2, 6.0)
)


# ----------------------------------------------------------
# Observed peak LOD
# ----------------------------------------------------------

ax.scatter(
    x,
    figure7_data["lod"],
    s=80,
    zorder=5,
    label="Observed peak LOD"
)


# ----------------------------------------------------------
# 10% empirical threshold
# ----------------------------------------------------------

ax.scatter(
    x,
    figure7_data["lod_threshold_10pct"],
    marker="_",
    s=450,
    linewidth=2.2,
    zorder=4,
    label="10% empirical threshold"
)


# ----------------------------------------------------------
# 5% empirical threshold
# ----------------------------------------------------------

ax.scatter(
    x,
    figure7_data["lod_threshold_05pct"],
    marker="_",
    s=450,
    linewidth=2.2,
    zorder=4,
    label="5% empirical threshold"
)


# ----------------------------------------------------------
# Connect observed point to 10% threshold
# ----------------------------------------------------------

for i, row in figure7_data.iterrows():

    ax.plot(
        [i, i],
        [
            row["lod"],
            row["lod_threshold_10pct"]
        ],
        linestyle=":",
        linewidth=0.8,
        alpha=0.55,
        zorder=1
    )

# ----------------------------------------------------------
# Observed LOD labels
#
# Put values below observed points.
# SCN FI3 needs extra downward offset because its
# 10% threshold lies immediately below the observed point.
# ----------------------------------------------------------

for i, row in figure7_data.iterrows():

    # Extra downward spacing for SCN FI3
    if row["trait"] == "scn_fi3":
        y_offset = -25
    else:
        y_offset = -11

    ax.annotate(
        f"{row['lod']:.2f}",
        (
            i,
            row["lod"]
        ),
        xytext=(0, y_offset),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=8.2,
        zorder=6
    )

# ----------------------------------------------------------
# X axis
# ----------------------------------------------------------

ax.set_xticks(x)

ax.set_xticklabels(
    figure7_data["label"],
    fontsize=8.5
)


ax.set_xlabel(
    "Disease phenotype and peak marker"
)


# ----------------------------------------------------------
# Y axis
# ----------------------------------------------------------

ax.set_ylabel(
    "LOD score"
)


ymax = (
    figure7_data[
        [
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct"
        ]
    ]
    .max()
    .max()
)


ax.set_ylim(
    0,
    ymax * 1.20
)


# ----------------------------------------------------------
# Title
# ----------------------------------------------------------

ax.set_title(
    "Genome-wide QTL evidence for SCN and SDS phenotypes",
    pad=42
)


# ----------------------------------------------------------
# Grid
# ----------------------------------------------------------

ax.grid(
    axis="y",
    alpha=0.18
)


# ----------------------------------------------------------
# Legend above plot
# ----------------------------------------------------------

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=3,
    frameon=False,
    fontsize=8.5
)


# ----------------------------------------------------------
# Interpretation note
# ----------------------------------------------------------

ax.text(
    0.985,
    0.955,
    (
        "Only SCN FI3 exceeded its\n"
        "10% genome-wide threshold."
    ),
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=7.8,
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        alpha=0.90,
        linewidth=0.5
    )
)


# ----------------------------------------------------------
# Layout
# ----------------------------------------------------------

fig.subplots_adjust(
    top=0.78,
    bottom=0.17,
    left=0.09,
    right=0.98
)


# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

save_manuscript_figure(
    fig,
    "Figure7_disease_qtl_evidence"
)


plt.show()
plt.close(fig)


# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print("DISEASE QTL FIGURE SUMMARY")
print("=" * 100)

display(
    figure7_data[
        [
            "trait",
            "peak_marker",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
)

### Cell 11.17 — Figure S5: recurrent SDS signal at Satt334

# Cell 11.17 — ROBUST REVISED VERSION
# Figure S5 — Recurrent subthreshold SDS signal near Satt334
#
# Tries, in order:
#   1. frozen QTL workbook sheets containing SDS regional profiles
#   2. Notebook 10 saved regional-profile source CSVs
#
# Saves:
#   source CSV
#   PNG
#   PDF
#
# Scientific interpretation:
#   Satt334 recurs as the peak for sds_ds, sds_dx, and sds_dx_mean,
#   but none reached the trait-specific 10% genome-wide threshold.

SDS_TRAITS = [
    "sds_ds",
    "sds_dx",
    "sds_dx_mean"
]


# ----------------------------------------------------------
# Inspect workbook sheets
# ----------------------------------------------------------

qtl_xls = pd.ExcelFile(
    QTL_FILE
)

print("QTL WORKBOOK SHEETS")
print("=" * 100)
print(qtl_xls.sheet_names)


# ----------------------------------------------------------
# Helper to detect cM column
# ----------------------------------------------------------

def detect_cm_column(df):

    candidates = [
        "kosambi_cm_structural_provisional",
        "kosambi_cm_provisional",
        "kosambi_cm",
        "position_cm",
        "cm"
    ]

    for col in candidates:
        if col in df.columns:
            return col

    return None


# ----------------------------------------------------------
# Try to recover SDS profiles from workbook sheets
# ----------------------------------------------------------

sds_profiles_list = []


for trait in SDS_TRAITS:

    found = False

    # Look for any sheet name containing the trait string
    matching_sheets = [
        sheet
        for sheet in qtl_xls.sheet_names
        if trait.lower() in sheet.lower()
    ]

    for sheet in matching_sheets:

        tmp = pd.read_excel(
            QTL_FILE,
            sheet_name=sheet
        )

        if (
            "marker" in tmp.columns
            and "lod" in tmp.columns
            and detect_cm_column(tmp) is not None
        ):

            tmp = tmp.copy()
            tmp["trait"] = trait
            tmp["_source"] = f"workbook:{sheet}"

            sds_profiles_list.append(
                tmp
            )

            print(
                f"{trait}: recovered from workbook sheet '{sheet}'"
            )

            found = True
            break


    # ------------------------------------------------------
    # Fallback to Notebook 10 saved source CSV
    # ------------------------------------------------------

    if not found:

        fallback_csv = (
            TABLE_DIR
            / f"figure10_15_{trait}_regional_profile_source.csv"
        )

        if fallback_csv.exists():

            tmp = pd.read_csv(
                fallback_csv
            )

            if (
                "marker" not in tmp.columns
                or "lod" not in tmp.columns
                or detect_cm_column(tmp) is None
            ):
                raise ValueError(
                    f"{trait}: fallback CSV exists but required columns "
                    f"are missing:\n{fallback_csv}"
                )

            tmp = tmp.copy()
            tmp["trait"] = trait
            tmp["_source"] = str(fallback_csv)

            sds_profiles_list.append(
                tmp
            )

            print(
                f"{trait}: recovered from fallback CSV"
            )

            found = True


    if not found:

        raise FileNotFoundError(
            f"Could not recover a regional profile for {trait} "
            "from either the frozen QTL workbook or Notebook 10 source CSVs."
        )


# ----------------------------------------------------------
# Combine profiles
# ----------------------------------------------------------

sds_profiles = pd.concat(
    sds_profiles_list,
    ignore_index=True,
    sort=False
)


# ----------------------------------------------------------
# Detect common cM column
# ----------------------------------------------------------

cm_col = detect_cm_column(
    sds_profiles
)

if cm_col is None:
    raise KeyError(
        "Could not identify SDS genetic-position column."
    )


# ----------------------------------------------------------
# Attach trait-specific 10% thresholds
# ----------------------------------------------------------

threshold_lookup = (
    disease_qtl
    .set_index("trait")[
        "lod_threshold_10pct"
    ]
    .to_dict()
)


sds_profiles[
    "lod_threshold_10pct"
] = sds_profiles[
    "trait"
].map(threshold_lookup)


# ----------------------------------------------------------
# Validate recurrent Satt334 peak
# ----------------------------------------------------------

peak_check = []

for trait in SDS_TRAITS:

    tmp = (
        sds_profiles
        .loc[
            sds_profiles["trait"] == trait
        ]
        .copy()
    )

    if len(tmp) == 0:
        continue

    peak_row = (
        tmp
        .sort_values(
            "lod",
            ascending=False
        )
        .iloc[0]
    )

    peak_check.append(
        {
            "trait": trait,
            "peak_marker_from_profile":
                peak_row["marker"],
            "peak_lod_from_profile":
                peak_row["lod"]
        }
    )


peak_check_df = pd.DataFrame(
    peak_check
)


print()
print("SDS REGIONAL PEAK CHECK")
print("=" * 100)

display(
    peak_check_df
)


# ----------------------------------------------------------
# Save source table
# ----------------------------------------------------------

save_figure_source_table(
    sds_profiles,
    "FigureS5_sds_satt334_regional_profiles_source"
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9.0, 5.8)
)


for trait in SDS_TRAITS:

    tmp = (
        sds_profiles
        .loc[
            sds_profiles["trait"] == trait
        ]
        .sort_values(cm_col)
    )

    ax.plot(
        tmp[cm_col],
        tmp["lod"],
        marker="o",
        linewidth=1.5,
        label=trait
    )


# ----------------------------------------------------------
# Mark recurrent Satt334 position
# ----------------------------------------------------------

satt334_rows = (
    sds_profiles
    .loc[
        sds_profiles["marker"].astype(str)
        == "Satt334"
    ]
)


if len(satt334_rows) > 0:

    satt334_cm = float(
        satt334_rows.iloc[0][cm_col]
    )

    ax.axvline(
        satt334_cm,
        linestyle=":",
        linewidth=1.1
    )

    max_satt334_lod = float(
        satt334_rows["lod"].max()
    )

    ax.annotate(
        "Satt334",
        (
            satt334_cm,
            max_satt334_lod
        ),
        xytext=(6, 10),
        textcoords="offset points",
        fontsize=8.5
    )


# ----------------------------------------------------------
# Axes
# ----------------------------------------------------------

ax.set_xlabel(
    "Provisional Kosambi position on pLG18 (cM)"
)

ax.set_ylabel(
    "Single-marker LOD"
)

ax.set_title(
    "Recurrent subthreshold SDS signal near Satt334"
)


ax.grid(
    alpha=0.18
)


ax.legend(
    frameon=False,
    loc="upper left"
)


# ----------------------------------------------------------
# Interpretation note
# ----------------------------------------------------------

ax.text(
    0.985,
    0.955,
    (
        "Satt334 was the peak for DS, DX, and DX mean;\n"
        "none reached the 10% genome-wide threshold."
    ),
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=7.8,
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        alpha=0.90,
        linewidth=0.5
    )
)


fig.subplots_adjust(
    top=0.90,
    bottom=0.15,
    left=0.10,
    right=0.98
)


# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

save_manuscript_figure(
    fig,
    "FigureS5_sds_satt334_regional_profiles"
)

plt.show()
plt.close(fig)


# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()
print("SDS FIGURE SUMMARY")
print("=" * 100)

display(
    disease_qtl.loc[
        disease_qtl["trait"].isin(
            SDS_TRAITS
        ),
        [
            "trait",
            "peak_marker",
            "lod",
            "lod_threshold_10pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
)

### Cell 11.18 — final figure manifest

In [ ]:
# Cell 11.18
# Final Notebook 11 figure/source-data manifest.
#
# Figure S5 is intentionally omitted because marker-by-marker
# regional SDS profiles were not preserved in the frozen outputs.
# No QTL scans are rerun in this manuscript-figure notebook.

manifest_records = []


# ----------------------------------------------------------
# Figures
# ----------------------------------------------------------

for path in sorted(
    MANUSCRIPT_FIGURE_DIR.glob("*")
):

    if path.is_file():

        manifest_records.append(
            {
                "category": "figure",
                "name": path.name,
                "extension": path.suffix.lower(),
                "path": str(path),
                "size_kb": round(
                    path.stat().st_size / 1024,
                    2
                )
            }
        )


# ----------------------------------------------------------
# Figure source tables
# ----------------------------------------------------------

for path in sorted(
    MANUSCRIPT_FIGURE_TABLE_DIR.glob("*")
):

    if path.is_file():

        manifest_records.append(
            {
                "category": "source_table",
                "name": path.name,
                "extension": path.suffix.lower(),
                "path": str(path),
                "size_kb": round(
                    path.stat().st_size / 1024,
                    2
                )
            }
        )


figure_manifest = pd.DataFrame(
    manifest_records
)


FIGURE_MANIFEST_CSV = (
    TABLE_DIR
    / "notebook11_manuscript_figure_manifest.csv"
)


figure_manifest.to_csv(
    FIGURE_MANIFEST_CSV,
    index=False
)


print("NOTEBOOK 11 MANUSCRIPT FIGURE MANIFEST")
print("=" * 100)

print()
print("Files by category:")
print(
    figure_manifest[
        "category"
    ]
    .value_counts()
)

print()
print("Files by extension:")
print(
    figure_manifest[
        "extension"
    ]
    .value_counts()
)

print()

display(
    figure_manifest
)

print()
print("Saved:")
print(FIGURE_MANIFEST_CSV)

### Cell 11.19 — final Notebook 11 validation
* This is the checkpoint I would use before freezing.

In [ ]:
# Patch for Figure 7 source-table filename
# No analysis is rerun.

FIGURE7_SOURCE_FIXED = (
    MANUSCRIPT_FIGURE_TABLE_DIR
    / "Figure7_disease_qtl_evidence_source.csv"
)


disease_qtl.to_csv(
    FIGURE7_SOURCE_FIXED,
    index=False
)


print("Figure 7 source table saved:")
print(FIGURE7_SOURCE_FIXED)

print()
print(
    "Exists:",
    FIGURE7_SOURCE_FIXED.exists()
)

In [ ]:
# Cell 11.19
# Final Notebook 11 validation checkpoint.
#
# Expected:
#   7 main figures
#   4 supplementary figures
#
# Total = 11 figures
#
# Figure S5 is intentionally not required because no frozen
# marker-by-marker SDS regional profile was available.

expected_figure_stems = [

    # Main figures
    "Figure1_structural_map_overview",
    "Figure2_four_suggestive_qtl_thresholds",
    "Figure3_scn_fi3_gm20_regional_profile",
    "Figure4_scn_fi3_allele_effect_profile",
    "Figure5_scn_gm20_physical_bracket",
    "Figure6_scn_gm20_high_priority_candidates",
    "Figure7_disease_qtl_evidence",

    # Supplementary figures
    "FigureS1_lrn_gm02_candidate_region",
    "FigureS2_prot03_gm20_anchor_context",
    "FigureS3_days_fl07_single_anchor_context",
    "FigureS4_qtl_physical_resolution_hierarchy"
]


validation_records = []


for stem in expected_figure_stems:

    png_path = (
        MANUSCRIPT_FIGURE_DIR
        / f"{stem}.png"
    )

    pdf_path = (
        MANUSCRIPT_FIGURE_DIR
        / f"{stem}.pdf"
    )


    # Figure source tables may have "_source" in filename.
    source_candidates = list(
        MANUSCRIPT_FIGURE_TABLE_DIR.glob(
            f"{stem}*_source.csv"
        )
    )


    validation_records.append(
        {
            "figure": stem,
            "png_exists": png_path.exists(),
            "pdf_exists": pdf_path.exists(),
            "source_table_exists":
                len(source_candidates) > 0
        }
    )


notebook11_validation = pd.DataFrame(
    validation_records
)


notebook11_validation[
    "all_required_files_present"
] = (
    notebook11_validation[
        [
            "png_exists",
            "pdf_exists",
            "source_table_exists"
        ]
    ]
    .all(axis=1)
)


VALIDATION_CSV = (
    TABLE_DIR
    / "notebook11_final_validation.csv"
)


notebook11_validation.to_csv(
    VALIDATION_CSV,
    index=False
)


print("NOTEBOOK 11 FINAL VALIDATION")
print("=" * 100)

display(
    notebook11_validation
)


print()
print(
    "Figures expected:",
    len(
        notebook11_validation
    )
)

print(
    "Figures fully complete:",
    int(
        notebook11_validation[
            "all_required_files_present"
        ].sum()
    )
)

print(
    "Notebook 11 ready to freeze:",
    bool(
        notebook11_validation[
            "all_required_files_present"
        ].all()
    )
)

print()
print("Validation saved:")
print(VALIDATION_CSV)